# Import lib etc

In [1]:
pip install "kagglehub[pandas-datasets]"

Note: you may need to restart the kernel to use updated packages.


Import dataframe & pick only 3 games

In [1]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "kieranpoc/steam-reviews", "weighted_score_above_08.csv")

100%|██████████| 299M/299M [00:09<00:00, 34.4MB/s] 

Extracting zip of weighted_score_above_08.csv...



/opt/anaconda3/lib/python3.12/site-packages/kagglehub/pandas_datasets.py:92: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  result = read_function(


In [4]:
games = [
    "Counter-Strike 2",
    "PUBG: BATTLEGROUNDS",
    "Grand Theft Auto V"
]

df_3games = df[df["game"].isin(games)].copy()

print(df_3games.shape)
df_3games["game"].value_counts()

(7365, 24)


game
Counter-Strike 2       3017
PUBG: BATTLEGROUNDS    2404
Grand Theft Auto V     1944
Name: count, dtype: int64

# 1. Prepare Gemini Labeling Prompt


In [ ]:
with open("../prompts/codebook_exact.txt", "r") as f:
    SYSTEM = f.read()

GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0

SCHEMA = {
    "type": "object",
    "properties": {
        "harm_mention": {"type": "string", "enum": ["yes", "no", "uncertain"]},
        "harm_reason": {"type": "string"},
        "irony": {"type": "string", "enum": ["yes", "no", "uncertain"]},
        "irony_reason": {"type": "string"},
        "gender_age": {"type": "string", "enum": ["none", "gender", "age", "both", "uncertain"]},
        "gender_age_reason": {"type": "string"}
    },
    "required": [
        "harm_mention", "harm_reason",
        "irony", "irony_reason",
        "gender_age", "gender_age_reason"
    ]
}


# 2. Set Up Gemini API and Labeling Function


In [ ]:
pip install google-genai python-dotenv


In [ ]:
import os
import json
import time
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types, errors
import random

load_dotenv("../.env")
# Set USE_VERTEX=1 in .env to go through Google Cloud (Vertex AI) instead of an AI Studio API key.
# Needs: `gcloud auth application-default login` + Vertex AI API enabled on the project.
if os.environ.get("USE_VERTEX") == "1":
    client = genai.Client(
        vertexai=True,
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.environ.get("GOOGLE_CLOUD_LOCATION", "global"),
    )
else:
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

def make_payload(row):
    if "recommendation" in row.index:
        recommendation = row["recommendation"]
    elif "voted_up" in row.index:
        recommendation = row["voted_up"]
    else:
        raise KeyError("No recommendation column found")

    if isinstance(recommendation, bool):
        recommendation = "Recommended" if recommendation else "Not recommended"

    return {
        "game": row["game"],
        "review": row["review"],
        "recommendation": recommendation
    }

RETRY_CODES = {429, 500, 503, 504}

def label_review(row, max_retries=8):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=json.dumps(make_payload(row), ensure_ascii=False),
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM,
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                    response_schema=SCHEMA
                )
            )
            return json.loads(response.text)

        except errors.APIError as e:
            # only retry capacity/rate-limit errors; exponential backoff + jitter
            if e.code not in RETRY_CODES or attempt == max_retries - 1:
                raise
            wait = min(10 * 2 ** attempt, 300) + random.uniform(0, 5)
            print(f"Attempt {attempt + 1} failed ({e.code}); retrying in {wait:.0f}s")
            time.sleep(wait)


# 3. Trial Batch on Human Positive Cases

Test Gemini on 10 reviews where Annotator 1 labeled toxicity or irony as `yes`, then compare the human and Gemini labels.


In [ ]:
annotator_1 = pd.read_csv("../data/blind_labelling_annotator_1.csv")

positive_cases = annotator_1[
    (annotator_1["toxicity_clean"] == "yes") |
    (annotator_1["irony_clean"] == "yes")
].copy()

trial_10 = positive_cases.sample(n=10, random_state=211).copy()

trial_10[["case_id", "game", "review", "recommendation", "toxicity_clean", "irony_clean"]]


In [ ]:
from IPython.display import clear_output

MODEL_TAG = GEMINI_MODEL.replace("/", "_")
CACHE = f"../data/gemini_trial_10_{MODEL_TAG}.jsonl"   # results are kept per model
PAUSE = 0.5   # seconds between requests (was 4.2)

done = {}
if os.path.exists(CACHE):
    with open(CACHE) as f:
        done = {json.loads(l)["case_id"]: json.loads(l) for l in f}

total = len(trial_10)
recent = []
n_new = 0
t0 = time.time()

def snip(text, n=110):
    text = str(text).replace("\n", " ")
    return text if len(text) <= n else text[:n] + "..."

def show(current=None):
    n_done = len(done)
    pct = n_done / total
    filled = int(pct * 30)
    bar = "█" * filled + "░" * (30 - filled)
    elapsed = time.time() - t0
    eta = (elapsed / n_new) * (total - n_done) if n_new else 0
    clear_output(wait=True)
    print(f"Model: {GEMINI_MODEL}")
    print(f"[{bar}] {pct*100:5.1f}%  ({n_done}/{total})  elapsed {elapsed:.0f}s  ETA ~{eta:.0f}s\n")
    for it in recent[-2:]:
        tox = "SAME" if it["human_tox"] == it["gem_tox"] else "DIFF"
        iro = "SAME" if it["human_iro"] == it["gem_iro"] else "DIFF"
        print(f"✔ SOLVED  {it['case_id']} | {it['game']}")
        print(f"   \"{snip(it['review'])}\"")
        print(f"   Gemini: toxicity={it['gem_tox']}  irony={it['gem_iro']}")
        print(f"   Human : toxicity={it['human_tox']}  irony={it['human_iro']}   -> tox {tox}, irony {iro}\n")
    if current is not None:
        print(f"⏳ PROCESSING  {current['case_id']} | {current['game']}")
        print(f"   \"{snip(current['review'])}\"\n")

show()
for _, row in trial_10.iterrows():
    cid = row["case_id"]
    if cid in done:
        continue
    show(current=row)
    result = label_review(row)
    entry = {"case_id": cid, "model": GEMINI_MODEL, **result}
    with open(CACHE, "a") as f:
        f.write(json.dumps(entry) + "\n")
    done[cid] = entry
    n_new += 1
    recent.append({
        "case_id": cid, "game": row["game"], "review": row["review"],
        "gem_tox": result["harm_mention"], "gem_iro": result["irony"],
        "human_tox": row["toxicity_clean"], "human_iro": row["irony_clean"],
    })
    show()
    time.sleep(PAUSE)

print(f"DONE: {len(done)}/{total} in {time.time() - t0:.0f}s")

rows = []
for _, row in trial_10.iterrows():
    result = done[row["case_id"]]
    rows.append({
        "case_id": row["case_id"],
        "model": GEMINI_MODEL,
        "game": row["game"],
        "review": row["review"],
        "recommendation": row["recommendation"],
        "human_toxicity": row["toxicity_clean"],
        "gemini_toxicity": result["harm_mention"],
        "human_irony": row["irony_clean"],
        "gemini_irony": result["irony"],
        "harm_reason": result["harm_reason"],
        "irony_reason": result["irony_reason"],
    })

trial_results = pd.DataFrame(rows)
trial_results["toxicity_match"] = trial_results["human_toxicity"] == trial_results["gemini_toxicity"]
trial_results["irony_match"] = trial_results["human_irony"] == trial_results["gemini_irony"]
trial_results.to_csv(f"../data/gemini_trial_10_{MODEL_TAG}.csv", index=False)
trial_results

In [ ]:
print("Toxicity matches:", trial_results["toxicity_match"].sum(), "/", len(trial_results))
print("Irony matches:", trial_results["irony_match"].sum(), "/", len(trial_results))


# 4. Scale-up: label every review

Run cells 4a, 4b and 4c in order (cells 1-3 and the setup cells above must have been run in this kernel first: `df_3games`, the prompt, the client, `make_payload`).

- **Resumable:** every finished review is appended to `../data/gemini_full_<model>.jsonl`. Re-running skips anything already stored, so nothing is paid for twice and a crash or interrupt loses nothing.
- **Failures don't stop the run:** a review that still fails after the retries is logged and shown in the panel; re-run 4b to retry only those.
- **First try a small run:** set `RUN_LIMIT = 200` in 4a. Set it back to `None` for the full run.
- Interrupt the kernel at any time to pause; run 4b again to continue.

In [ ]:
import hashlib, html, json, os, queue, random, threading, time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from google.genai import errors
from IPython.display import HTML, display

# ============================ settings ============================
BATCH_SIZE = 100      # display + progress only; every review is saved to disk the moment it finishes
WORKERS = 8           # parallel requests
RUN_LIMIT = None      # e.g. 200 for a quick dry run; None = all reviews
RETRY_CODES = {429, 500, 503, 504}
BACKOFF_BASE, BACKOFF_MAX = 10, 300   # seconds (exponential backoff + jitter)
# ==================================================================

MODEL_TAG = GEMINI_MODEL.replace("/", "_")
BACKEND = "Vertex AI" if os.environ.get("USE_VERTEX") == "1" else "AI Studio"
FULL_CACHE = f"../data/gemini_full_{MODEL_TAG}.jsonl"          # one line per finished review
ERR_LOG = f"../data/gemini_full_{MODEL_TAG}_errors.jsonl"      # failures, for the record


def _clean_row(row):
    r = row.copy()
    if "recommendation" not in r.index and "voted_up" in r.index:
        r["voted_up"] = bool(r["voted_up"])      # numpy bool -> plain bool so json.dumps works
    return r


def _review_ok(row):
    return isinstance(row["review"], str) and row["review"].strip() != ""


def _key(idx, row):
    # row index + short hash of game/review: if the dataset ever changes, old cache entries stop matching
    h = hashlib.sha1(f'{row["game"]}|{row["review"]}'.encode("utf-8")).hexdigest()[:8]
    return f"{idx}-{h}"


def _jsonable(x):
    return x.item() if hasattr(x, "item") else x


jobs = []
for pos, (idx, row) in enumerate(df_3games.iterrows()):
    jobs.append({"pos": pos, "idx": idx, "key": _key(idx, row), "game": row["game"], "row": _clean_row(row)})
if RUN_LIMIT:
    jobs = jobs[:RUN_LIMIT]

# lookup: what is already stored? (these get skipped, so nothing is ever paid for twice)
results = {}
if os.path.exists(FULL_CACHE):
    with open(FULL_CACHE, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                e = json.loads(line)
                results[e["key"]] = e
already = sum(1 for j in jobs if j["key"] in results)

net = {"retries": 0, "waiting": 0}
net_lock = threading.Lock()


def label_quiet(row, max_retries=8):
    """Same call as label_review, but silent (the live panel shows retries) and thread-safe."""
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=json.dumps(make_payload(row), ensure_ascii=False),
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM,
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                    response_schema=SCHEMA,
                ),
            )
            if not response.text:
                raise ValueError("empty response (blocked by a safety filter, or no text returned)")
            return json.loads(response.text)
        except errors.APIError as e:
            if e.code not in RETRY_CODES or attempt == max_retries - 1:
                raise
            wait = min(BACKOFF_BASE * 2 ** attempt, BACKOFF_MAX) + random.uniform(0, BACKOFF_BASE / 2)
            with net_lock:
                net["retries"] += 1
                net["waiting"] += 1
            try:
                time.sleep(wait)
            finally:
                with net_lock:
                    net["waiting"] -= 1


print(f"{BACKEND} · {GEMINI_MODEL}")
print(f"{len(jobs):,} reviews in scope · {already:,} already labeled in {FULL_CACHE} (skipped) · {len(jobs) - already:,} to go")

In [ ]:
_CSS = """
<style>
.sr{--surface:#191919;--panel:#242424;--line:#343434;--tx:#efefef;--tx2:#9c9c9c;--blue:#338ced;--blue-lt:#6aabf1;--orange:#e76624;--neutral:#5f5f5f;--crit:#dc3c3d;--good:#0aac23;--warn:#fbb315;
 font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif;background:var(--surface);color:var(--tx);border:1px solid var(--line);border-radius:18px;padding:22px 24px;max-width:980px;box-sizing:border-box;line-height:1.35}
.sr *{box-sizing:border-box}
.sr-head{display:flex;justify-content:space-between;align-items:flex-start;gap:12px;flex-wrap:wrap}
.sr-title{font-size:15px;font-weight:600}
.sr-sub{font-size:12px;color:var(--tx2);margin-top:3px}
.sr-chip{display:inline-flex;align-items:center;gap:8px;font-size:11px;font-weight:700;letter-spacing:.07em;padding:6px 12px;border-radius:999px;border:1px solid var(--line);background:var(--panel)}
.sr-dot{width:9px;height:9px;border-radius:50%;background:var(--blue)}
.sr-run .sr-dot{animation:sr-blink 1.1s ease-in-out infinite}
.sr-done .sr-dot{background:var(--good)}
.sr-warn .sr-dot{background:var(--crit)}
.sr-pause .sr-dot{background:var(--warn)}
.sr-hero{display:flex;align-items:baseline;gap:14px;margin:20px 0 12px;flex-wrap:wrap}
.sr-pct{font-size:50px;font-weight:700;line-height:1;font-variant-numeric:tabular-nums;letter-spacing:-.02em}
.sr-of{font-size:13px;color:var(--tx2);font-variant-numeric:tabular-nums}
.sr-bar{height:14px;border-radius:999px;background:var(--panel);overflow:hidden}
.sr-fill{height:100%;border-radius:999px;background:var(--blue);transition:width .45s ease}
.sr-run .sr-fill{background:linear-gradient(90deg,var(--blue) 0%,var(--blue-lt) 50%,var(--blue) 100%);background-size:220% 100%;animation:sr-shine 1.7s linear infinite}
.sr-bar.thin{height:8px;margin-top:8px}
.sr-stats{display:grid;grid-template-columns:repeat(5,minmax(0,1fr));gap:10px;margin:18px 0 6px}
.sr-stat{background:var(--panel);border-radius:12px;padding:11px 13px}
.sr-k{font-size:10.5px;letter-spacing:.07em;text-transform:uppercase;color:var(--tx2)}
.sr-v{font-size:19px;font-weight:600;margin-top:3px;font-variant-numeric:tabular-nums;white-space:nowrap}
.sr-v small{font-size:11px;font-weight:500;color:var(--tx2)}
.sr-h{font-size:12px;font-weight:600;margin:22px 0 9px;display:flex;justify-content:space-between;gap:10px;flex-wrap:wrap}
.sr-h span{font-weight:400;color:var(--tx2)}
.sr-tiles{display:grid;grid-template-columns:repeat(auto-fill,minmax(24px,1fr));gap:3px}
.sr-t{aspect-ratio:1;border-radius:6px;background:var(--panel);position:relative;overflow:hidden}
.sr-t i{position:absolute;left:0;right:0;bottom:0;background:var(--blue);display:block}
.sr-t.full i{height:100%}
.sr-t.part.live i{animation:sr-pulse 1.1s ease-in-out infinite}
.sr-t.bad{background:var(--crit);color:#fff;font:700 12px/24px sans-serif;text-align:center}
.sr-legend{display:flex;gap:16px;flex-wrap:wrap;font-size:11px;color:var(--tx2);margin-top:9px}
.sr-legend b{display:inline-block;width:10px;height:10px;border-radius:3px;margin-right:6px;vertical-align:-1px}
.sr-cols{display:grid;grid-template-columns:1fr 1fr;gap:26px}
.sr-row{margin-bottom:12px}
.sr-rowh{display:flex;justify-content:space-between;font-size:12px;margin-bottom:6px;font-variant-numeric:tabular-nums}
.sr-rowh span{color:var(--tx2)}
.sr-stack{display:flex;height:12px;border-radius:999px;overflow:hidden;background:var(--panel)}
.sr-stack div{height:100%;border-left:2px solid var(--surface);transition:width .45s ease}
.sr-stack div:first-child{border-left:0}
.sr-key{display:flex;gap:14px;flex-wrap:wrap;font-size:11px;color:var(--tx2);margin-top:6px;font-variant-numeric:tabular-nums}
.sr-key b{display:inline-block;width:8px;height:8px;border-radius:50%;margin-right:5px}
.sr-feed{display:flex;flex-direction:column;gap:6px}
.sr-item{display:grid;grid-template-columns:54px 1fr auto;gap:12px;align-items:center;background:var(--panel);border-radius:10px;padding:8px 12px;font-size:12px}
.sr-game{font-size:10.5px;font-weight:700;letter-spacing:.05em;color:var(--tx2)}
.sr-txt{white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.sr-badges{display:flex;gap:6px;font-size:11px;color:var(--tx2);white-space:nowrap}
.sr-badges span{display:inline-flex;align-items:center;gap:5px;border:1px solid var(--line);border-radius:999px;padding:2px 8px}
.sr-badges b{width:7px;height:7px;border-radius:50%;display:inline-block}
.sr-note{font-size:12px;color:var(--tx2);margin-top:14px}
@keyframes sr-blink{0%,100%{opacity:1}50%{opacity:.25}}
@keyframes sr-shine{from{background-position:200% 0}to{background-position:-20% 0}}
@keyframes sr-pulse{0%,100%{opacity:1}50%{opacity:.55}}
@media (prefers-reduced-motion:reduce){.sr *{animation:none!important;transition:none!important}}
@media (max-width:720px){.sr-stats{grid-template-columns:repeat(2,1fr)}.sr-cols{grid-template-columns:1fr}}
</style>
"""

_SHORT = {"Counter-Strike 2": "CS2", "PUBG: BATTLEGROUNDS": "PUBG", "Grand Theft Auto V": "GTA V"}
_COLOR = {"yes": "var(--blue)", "uncertain": "var(--orange)", "no": "var(--neutral)"}

n_batches = (len(jobs) + BATCH_SIZE - 1) // BATCH_SIZE
b_total, b_done, b_fail = [0] * n_batches, [0] * n_batches, [0] * n_batches
games = sorted({j["game"] for j in jobs})
g_total, g_done = {g: 0 for g in games}, {g: 0 for g in games}
mix = {"harm_mention": {"yes": 0, "uncertain": 0, "no": 0}, "irony": {"yes": 0, "uncertain": 0, "no": 0}}
state = {"done": 0, "skipped": 0}
recent, stamps, failed = deque(maxlen=4), deque(), {}
for j in jobs:
    b_total[j["pos"] // BATCH_SIZE] += 1
    g_total[j["game"]] += 1


def _register(job, entry, fresh):
    b = job["pos"] // BATCH_SIZE
    state["done"] += 1
    b_done[b] += 1
    g_done[job["game"]] += 1
    if entry.get("status") == "skipped_empty":
        state["skipped"] += 1
        return
    for fld in mix:
        v = entry.get(fld)
        mix[fld][v] = mix[fld].get(v, 0) + 1
    if fresh:
        recent.appendleft(entry | {"_review": job["row"]["review"]})
        stamps.append(time.time())


for j in jobs:                                   # what the cache already covers
    if j["key"] in results:
        _register(j, results[j["key"]], fresh=False)


def _fmt_clock(s):
    s = int(s)
    h, rem = divmod(s, 3600)
    m, sec = divmod(rem, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m:02d}:{sec:02d}"


def _fmt_eta(s):
    if s is None:
        return "calculating…"
    if s < 60:
        return f"~{int(s)} s"
    if s < 3600:
        return f"~{int(round(s / 60))} min"
    return f"~{int(s // 3600)}h {int(s % 3600 // 60):02d}m"


def _mix_block(title, counts):
    total = sum(counts.values()) or 1
    segs = "".join(f'<div style="width:{100 * counts[k] / total:.2f}%;background:{_COLOR[k]}"></div>' for k in ("yes", "uncertain", "no"))
    key = "".join(f'<span><b style="background:{_COLOR[k]}"></b>{k} {counts[k]:,}</span>' for k in ("yes", "uncertain", "no"))
    return f'<div class="sr-row"><div class="sr-rowh">{title}<span>{sum(counts.values()):,} labeled</span></div><div class="sr-stack">{segs}</div><div class="sr-key">{key}</div></div>'


def _badge(label, v):
    return f'<span><b style="background:{_COLOR.get(v, "var(--neutral)")}"></b>{label} {html.escape(str(v))}</span>'


def _render(status):
    now = time.time()
    elapsed = now - t_start
    while stamps and now - stamps[0] > 30:
        stamps.popleft()
    rate = len(stamps) / max(min(30.0, elapsed), 1e-9) if stamps else 0.0
    total, done, n_fail = len(jobs), state["done"], len(failed)
    remaining = max(total - done - n_fail, 0)
    pct = 100.0 * done / total if total else 100.0
    eta = None if (remaining and rate <= 0) else (remaining / rate if remaining else 0)

    if status == "RUNNING":
        cls, chip = "sr-run", "RUNNING"
    elif status == "PAUSED":
        cls, chip = "sr-pause", "PAUSED · re-run the cell to resume"
    elif n_fail:
        cls, chip = "sr-warn", f"! DONE · {n_fail} failed"
    else:
        cls, chip = "sr-done", "✓ DONE"

    live = "live" if status == "RUNNING" else ""

    def ph(period):   # the panel is re-drawn on every refresh; tying the animation phase to the clock keeps it smooth
        return f"animation-delay:-{now % period:.2f}s"

    tiles = []
    for b in range(n_batches):
        d, f, t = b_done[b], b_fail[b], b_total[b]
        tip = f"Batch {b + 1}: {d}/{t} ({100 * d // t}%)" + (f" · {f} failed" if f else "")
        if f and d + f == t:
            tiles.append(f'<div class="sr-t bad" title="{tip}">!</div>')
        elif d == t:
            tiles.append(f'<div class="sr-t full" title="{tip}"><i></i></div>')
        elif d:
            tiles.append(f'<div class="sr-t part {live}" title="{tip}"><i style="height:{100 * d / t:.0f}%;{ph(1.1)}"></i></div>')
        else:
            tiles.append(f'<div class="sr-t" title="{tip}"></div>')

    active = next((b for b in range(n_batches) if b_done[b] + b_fail[b] < b_total[b]), None)
    if active is None:
        cur_label, cur_pct = "All batches finished", 100.0
    else:
        cur_pct = 100.0 * b_done[active] / b_total[active]
        cur_label = f"Batch {active + 1} of {n_batches} · {b_done[active]}/{b_total[active]} · {cur_pct:.0f}%"

    game_rows = "".join(
        f'<div class="sr-row"><div class="sr-rowh">{html.escape(g)}<span>{g_done[g]:,} / {g_total[g]:,}</span></div>'
        f'<div class="sr-bar thin" style="margin-top:0"><div class="sr-fill" style="width:{100 * g_done[g] / max(g_total[g], 1):.1f}%"></div></div></div>'
        for g in games)

    feed = "".join(
        f'<div class="sr-item"><div class="sr-game">{_SHORT.get(e["game"], e["game"][:6])}</div>'
        f'<div class="sr-txt">{html.escape(" ".join(str(e["_review"]).split())[:140])}</div>'
        f'<div class="sr-badges">{_badge("toxicity", e.get("harm_mention"))}{_badge("irony", e.get("irony"))}</div></div>'
        for e in recent) or '<div class="sr-note">Waiting for the first results…</div>'

    waiting = f' <small>({net["waiting"]} waiting)</small>' if net["waiting"] else ""
    bad = f'<span style="color:var(--crit)">! </span>' if n_fail else ""
    return f"""{_CSS}
<div class="sr {cls}">
 <div class="sr-head">
  <div><div class="sr-title">Steam review labeling</div><div class="sr-sub">{html.escape(GEMINI_MODEL)} · {BACKEND} · {WORKERS} workers · batches of {BATCH_SIZE}</div></div>
  <div class="sr-chip"><i class="sr-dot" style="{ph(1.1)}"></i>{chip}</div>
 </div>
 <div class="sr-hero"><div class="sr-pct">{pct:.1f}%</div><div class="sr-of">{done:,} of {total:,} reviews labeled</div></div>
 <div class="sr-bar"><div class="sr-fill" style="width:{pct:.2f}%;{ph(1.7)}"></div></div>
 <div class="sr-stats">
  <div class="sr-stat"><div class="sr-k">Speed</div><div class="sr-v">{rate:.1f} <small>rev/s</small></div></div>
  <div class="sr-stat"><div class="sr-k">Time left</div><div class="sr-v">{_fmt_eta(eta) if remaining else "—"}</div></div>
  <div class="sr-stat"><div class="sr-k">Elapsed</div><div class="sr-v">{_fmt_clock(elapsed)}</div></div>
  <div class="sr-stat"><div class="sr-k">Retries (503/429)</div><div class="sr-v">{net["retries"]:,}{waiting}</div></div>
  <div class="sr-stat"><div class="sr-k">Failed</div><div class="sr-v">{bad}{n_fail:,}</div></div>
 </div>
 <div class="sr-h">Batches <span>{cur_label}</span></div>
 <div class="sr-tiles">{"".join(tiles)}</div>
 <div class="sr-bar thin"><div class="sr-fill" style="width:{cur_pct:.1f}%"></div></div>
 <div class="sr-legend"><span><b style="background:var(--panel);border:1px solid var(--line)"></b>pending</span><span><b style="background:var(--blue)"></b>in progress / done</span><span><b style="background:var(--crit)"></b>! has failures</span></div>
 <div class="sr-cols">
  <div><div class="sr-h">Labels so far</div>{_mix_block("Toxicity", mix["harm_mention"])}{_mix_block("Irony", mix["irony"])}</div>
  <div><div class="sr-h">By game</div>{game_rows}</div>
 </div>
 <div class="sr-h">Latest labeled</div>
 <div class="sr-feed">{feed}</div>
 {f'<div class="sr-note">{state["skipped"]} review(s) skipped (empty text).</div>' if state["skipped"] else ""}
</div>"""


resq = queue.Queue()


def _work(job):
    row = job["row"]
    try:
        if not _review_ok(row):
            res = {"ok": True, "entry": {"status": "skipped_empty"}}
        else:
            res = {"ok": True, "entry": {"status": "ok", **label_quiet(row)}}
    except Exception as e:
        res = {"ok": False, "error": f"{type(e).__name__}: {e}"}
    res["job"] = job
    resq.put(res)


def _handle(res):
    job = res["job"]
    if res["ok"]:
        entry = {"key": job["key"], "idx": _jsonable(job["idx"]), "game": job["game"],
                 "model": GEMINI_MODEL, "ts": round(time.time()), **res["entry"]}
        with open(FULL_CACHE, "a", encoding="utf-8") as f:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        results[job["key"]] = entry
        _register(job, entry, fresh=True)
    else:
        failed[job["key"]] = res["error"]
        b_fail[job["pos"] // BATCH_SIZE] += 1
        with open(ERR_LOG, "a", encoding="utf-8") as f:
            f.write(json.dumps({"key": job["key"], "error": res["error"], "ts": round(time.time())}) + "\n")


pending = [j for j in jobs if j["key"] not in results]
t_start = time.time()
status = "RUNNING"
handle = display(HTML(_render(status)), display_id=True)
ex = ThreadPoolExecutor(max_workers=WORKERS)
try:
    for j in pending:
        ex.submit(_work, j)
    got, last = 0, 0.0
    while got < len(pending):
        try:
            _handle(resq.get(timeout=0.5))
            got += 1
        except queue.Empty:
            pass
        if time.time() - last > 0.4:
            handle.update(HTML(_render("RUNNING")))
            last = time.time()
    status = "DONE"
except KeyboardInterrupt:
    status = "PAUSED"
finally:
    ex.shutdown(wait=False, cancel_futures=True)
    handle.update(HTML(_render(status)))

print(f"{status}: {state['done']:,}/{len(jobs):,} labeled · {len(failed)} failed · this run took {_fmt_clock(time.time() - t_start)}")
if failed:
    print("First errors:")
    for k, v in list(failed.items())[:3]:
        print("  ", k, "->", v[:200])
    print("Re-run this cell to retry only the failed ones (finished reviews are skipped).")
elif status == "PAUSED":
    print("Re-run this cell to continue where it stopped (finished reviews are skipped).")

In [ ]:
import pandas as pd

meta = pd.DataFrame([
    {"key": j["key"], "recommendation": make_payload(j["row"])["recommendation"], "review": j["row"]["review"]}
    for j in jobs if j["key"] in results
])
labels = pd.DataFrame([results[j["key"]] for j in jobs if j["key"] in results])
out = labels.merge(meta, on="key")
cols = ["key", "idx", "game", "recommendation", "review", "status",
        "harm_mention", "harm_reason", "irony", "irony_reason",
        "gender_age", "gender_age_reason", "model"]
out = out[[c for c in cols if c in out.columns]]
out.to_csv(f"../data/gemini_labels_{MODEL_TAG}.csv", index=False)
print(f"Saved {len(out):,} rows -> ../data/gemini_labels_{MODEL_TAG}.csv  ({len(jobs) - len(out):,} of {len(jobs):,} reviews not labeled yet)")

ok = out[out["status"] == "ok"]
print("\nToxicity by game");  display(pd.crosstab(ok["game"], ok["harm_mention"], margins=True))
print("Irony by game");      display(pd.crosstab(ok["game"], ok["irony"], margins=True))
print("Gender / age by game"); display(pd.crosstab(ok["game"], ok["gender_age"], margins=True))